# Step8 Task1: Readout Geometry Observation on Sequential SFT

This notebook reads sequential SFT runs from W&B, caches the history locally, extracts readout-geometry metrics into tidy pandas DataFrames, lists similar runs with annotations, and draws quick sanity-check curves.

Target run from the JD:

`multilingual_seq_qwen25_0p5b_rounds2_20epoch_lr1e4_constantlr_readoutgeom_20260810`


In [ ]:
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd

try:
    import wandb
except ImportError as exc:
    raise ImportError('Please install wandb in this environment before running this notebook.') from exc

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 140)


## Configuration

Change `ENTITY` if your W&B entity is different. Set `FORCE_REFRESH=True` when you want to re-download histories from W&B.

In [ ]:
ENTITY = '<WANDB_ENTITY>'
PROJECT = 'plasticity-loss'
TARGET_RUN_NAME = 'multilingual_seq_qwen25_0p5b_rounds2_20epoch_lr1e4_constantlr_readoutgeom_20260810'

CACHE_DIR = Path('cache/step8_task1_readout_geometry')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

FORCE_REFRESH = False
HISTORY_PAGE_SIZE = 1000

TASKS = ['gsm8k', 'mbpp', 'dolly_qa']
READOUT_METRICS = ['trace', 'effective_rank', 'hoyer_concentration']
DEFAULT_SUPPORT = 'raw_top300'


## W&B Helpers

The helpers below cache run metadata and histories locally. A history cache is keyed by W&B run id, so renaming a run will not invalidate old cached history.

In [ ]:
READOUT_RE = re.compile(r'^readout_geometry/([^/]+)/([^/]+)/(trace|effective_rank|hoyer_concentration|max_eigenvalue)$')


def sanitize_filename(text: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', text).strip('_')


def api() -> 'wandb.Api':
    return wandb.Api()


def run_path(run: Any) -> str:
    return f'{run.entity}/{run.project}/{run.id}'


def run_display_name(run: Any) -> str:
    return getattr(run, 'name', None) or getattr(run, 'display_name', None) or run.id


def config_get(config: dict[str, Any], *names: str, default: Any = None) -> Any:
    for name in names:
        if name in config and config[name] not in (None, ''):
            return config[name]
    return default


def list_project_runs(entity: str = ENTITY, project: str = PROJECT, *, force_refresh: bool = FORCE_REFRESH) -> pd.DataFrame:
    cache_path = CACHE_DIR / f'runs_{sanitize_filename(entity)}_{sanitize_filename(project)}.csv'
    if cache_path.exists() and not force_refresh:
        return pd.read_csv(cache_path)

    rows = []
    for run in api().runs(f'{entity}/{project}'):
        cfg = dict(run.config or {})
        summary_keys = list(dict(run.summary or {}).keys())
        readout_summary_keys = [k for k in summary_keys if k.startswith('readout_geometry/')]
        support_files = config_get(cfg, 'readout_geometry_support_files', 'READOUT_GEOMETRY_SUPPORT_FILES', default='')
        run_name = run_display_name(run)
        rows.append({
            'entity': run.entity,
            'project': run.project,
            'run_id': run.id,
            'name': run_name,
            'path': run_path(run),
            'state': run.state,
            'created_at': getattr(run, 'created_at', None),
            'url': run.url,
            'model_name': config_get(cfg, 'model_name', 'MODEL_NAME'),
            'subsets_dir': config_get(cfg, 'subsets_dir', 'SUBSETS_DIR'),
            'num_task_rounds': config_get(cfg, 'num_task_rounds', 'NUM_TASK_ROUNDS'),
            'num_train_epochs_per_task': config_get(cfg, 'num_train_epochs_per_task', 'NUM_TRAIN_EPOCHS_PER_TASK'),
            'learning_rate': config_get(cfg, 'learning_rate', 'LEARNING_RATE'),
            'lr_scheduler_type': config_get(cfg, 'lr_scheduler_type', 'LR_SCHEDULER_TYPE'),
            'warmup_ratio': config_get(cfg, 'warmup_ratio', 'WARMUP_RATIO'),
            'weight_decay': config_get(cfg, 'weight_decay', 'WEIGHT_DECAY'),
            'readout_geometry_logging_steps': config_get(cfg, 'readout_geometry_logging_steps', 'READOUT_GEOMETRY_LOGGING_STEPS'),
            'probe_eval_steps': config_get(cfg, 'probe_eval_steps', 'PROBE_EVAL_STEPS'),
            'support_files': support_files,
            'has_readout_geometry_summary': bool(readout_summary_keys),
            'readout_summary_key_count': len(readout_summary_keys),
            'annotation': make_run_annotation(run_name, cfg, support_files, readout_summary_keys),
        })

    df = pd.DataFrame(rows).sort_values(['created_at', 'name'], na_position='last').reset_index(drop=True)
    df.to_csv(cache_path, index=False)
    return df


def make_run_annotation(run_name: str, cfg: dict[str, Any], support_files: Any, readout_summary_keys: list[str]) -> str:
    text = ' '.join(str(x) for x in [run_name, support_files, config_get(cfg, 'run_id', 'RUN_ID', default='')]).lower()
    support = 'gradient' if 'gradient' in text else ('raw' if 'raw' in text else 'unknown_support')
    data = 'multilingual' if 'multilingual' in text or 'multilingual' in str(config_get(cfg, 'subsets_dir', 'SUBSETS_DIR', default='')).lower() else 'raw_or_unknown_data'
    scheduler = config_get(cfg, 'lr_scheduler_type', 'LR_SCHEDULER_TYPE', default='unknown_scheduler')
    epochs = config_get(cfg, 'num_train_epochs_per_task', 'NUM_TRAIN_EPOCHS_PER_TASK', default='?')
    rounds = config_get(cfg, 'num_task_rounds', 'NUM_TASK_ROUNDS', default='?')
    readout = 'readout_summary' if readout_summary_keys else 'readout_config_or_history_check_needed'
    return f'{data}; {support}; rounds={rounds}; epochs/task={epochs}; scheduler={scheduler}; {readout}'


def find_run_by_name(name: str, runs_df: pd.DataFrame) -> Any:
    matches = runs_df[runs_df['name'] == name]
    if matches.empty:
        partial = runs_df[runs_df['name'].str.contains(re.escape(name), case=False, na=False)]
        if partial.empty:
            raise ValueError(f'No W&B run found for name={name!r}. Try FORCE_REFRESH=True or check ENTITY/PROJECT.')
        warnings.warn(f'No exact run-name match. Using first partial match: {partial.iloc[-1]["name"]}')
        row = partial.iloc[-1]
    else:
        row = matches.iloc[-1]
    return api().run(row['path'])


def history_cache_path(run: Any) -> Path:
    return CACHE_DIR / f'history_{sanitize_filename(run.entity)}_{sanitize_filename(run.project)}_{run.id}.parquet'


def load_run_history(run: Any, *, force_refresh: bool = FORCE_REFRESH) -> pd.DataFrame:
    cache_path = history_cache_path(run)
    csv_fallback = cache_path.with_suffix('.csv')
    if cache_path.exists() and not force_refresh:
        return pd.read_parquet(cache_path)
    if csv_fallback.exists() and not force_refresh:
        return pd.read_csv(csv_fallback)

    rows = list(run.scan_history(page_size=HISTORY_PAGE_SIZE))
    df = pd.DataFrame(rows)
    if df.empty:
        warnings.warn(f'No history rows downloaded for {run_display_name(run)}')
    try:
        df.to_parquet(cache_path, index=False)
    except Exception as exc:
        warnings.warn(f'Could not write parquet cache ({exc}); writing CSV fallback instead.')
        df.to_csv(csv_fallback, index=False)
    return df


## List Similar Sequential SFT Runs with Readout Geometry

This table is intentionally broad. It includes runs whose name/config/summary suggests sequential SFT plus readout-geometry tracking.

In [ ]:
runs_df = list_project_runs()

name_blob = runs_df['name'].fillna('').str.lower()
support_blob = runs_df['support_files'].fillna('').astype(str).str.lower()
annotation_blob = runs_df['annotation'].fillna('').str.lower()

similar_runs = runs_df[
    (name_blob.str.contains('seq') | name_blob.str.contains('sequential') | name_blob.str.contains('long_seq'))
    & (
        runs_df['has_readout_geometry_summary']
        | support_blob.str.contains('top300|readout')
        | annotation_blob.str.contains('readout')
    )
].copy()

similar_runs = similar_runs.sort_values(['created_at', 'name'], na_position='last')
display_cols = [
    'created_at', 'state', 'name', 'model_name', 'subsets_dir', 'num_task_rounds',
    'num_train_epochs_per_task', 'learning_rate', 'lr_scheduler_type', 'warmup_ratio',
    'weight_decay', 'readout_geometry_logging_steps', 'probe_eval_steps', 'annotation', 'url'
]
print(f'Found {len(similar_runs)} similar sequential/readout-geometry runs.')
similar_runs[display_cols]


## Download and Tidy the Target Run

The tidy table has one row per evaluation point, support, task, and metric. The wide table has one row per W&B history row.

In [ ]:
target_run = find_run_by_name(TARGET_RUN_NAME, runs_df)
print('Target:', run_display_name(target_run), run_path(target_run))
print('URL:', target_run.url)

history_df = load_run_history(target_run)
print('History shape:', history_df.shape)
history_df.head()


In [ ]:
def readout_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in df.columns if READOUT_RE.match(str(col))]


def tidy_readout_history(history: pd.DataFrame, run: Any, runs_table: pd.DataFrame | None = None) -> pd.DataFrame:
    columns = readout_columns(history)
    if not columns:
        warnings.warn('No readout_geometry columns found in the history DataFrame.')
        return pd.DataFrame()

    id_vars = [col for col in ['_step', '_timestamp', 'continuous_epoch', 'task_name', 'stage_index', 'round_index', 'task_index'] if col in history.columns]
    tidy = history[id_vars + columns].melt(
        id_vars=id_vars,
        value_vars=columns,
        var_name='wandb_metric',
        value_name='value',
    ).dropna(subset=['value']).copy()

    parsed = tidy['wandb_metric'].str.extract(READOUT_RE)
    parsed.columns = ['support', 'support_task', 'metric']
    tidy = pd.concat([tidy, parsed], axis=1)
    tidy['run_id'] = run.id
    tidy['run_name'] = run_display_name(run)
    tidy['run_path'] = run_path(run)

    if runs_table is not None and run.id in set(runs_table['run_id']):
        meta = runs_table[runs_table['run_id'] == run.id].iloc[-1]
        for col in ['model_name', 'subsets_dir', 'num_task_rounds', 'num_train_epochs_per_task', 'learning_rate', 'lr_scheduler_type', 'warmup_ratio', 'weight_decay', 'annotation']:
            tidy[col] = meta.get(col)

    sort_cols = [col for col in ['continuous_epoch', '_step', 'support', 'support_task', 'metric'] if col in tidy.columns]
    return tidy.sort_values(sort_cols).reset_index(drop=True)


readout_tidy = tidy_readout_history(history_df, target_run, runs_df)
readout_wide = readout_tidy.pivot_table(
    index=[col for col in ['run_id', 'run_name', '_step', 'continuous_epoch', 'support', 'support_task'] if col in readout_tidy.columns],
    columns='metric',
    values='value',
    aggfunc='first',
).reset_index() if not readout_tidy.empty else pd.DataFrame()
readout_wide.columns.name = None

readout_tidy.to_csv(CACHE_DIR / f'tidy_readout_{sanitize_filename(target_run.id)}.csv', index=False)
readout_wide.to_csv(CACHE_DIR / f'wide_readout_{sanitize_filename(target_run.id)}.csv', index=False)

print('Readout columns:', len(readout_columns(history_df)))
print('Tidy shape:', readout_tidy.shape)
print('Wide shape:', readout_wide.shape)
display(readout_tidy.head(12))


## Filtering Utilities

Use these small functions to select support type, task support, metric, or epoch range before making your own figures.

In [ ]:
def filter_readout(
    df: pd.DataFrame,
    *,
    support: str | None = None,
    tasks: list[str] | None = None,
    metrics: list[str] | None = None,
    min_epoch: float | None = None,
    max_epoch: float | None = None,
) -> pd.DataFrame:
    out = df.copy()
    if support is not None:
        out = out[out['support'] == support]
    if tasks is not None:
        out = out[out['support_task'].isin(tasks)]
    if metrics is not None:
        out = out[out['metric'].isin(metrics)]
    if min_epoch is not None and 'continuous_epoch' in out.columns:
        out = out[out['continuous_epoch'] >= min_epoch]
    if max_epoch is not None and 'continuous_epoch' in out.columns:
        out = out[out['continuous_epoch'] <= max_epoch]
    return out.copy()


def summarize_readout(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return (
        df.groupby(['run_name', 'support', 'support_task', 'metric'], dropna=False)
        .agg(points=('value', 'count'), first_epoch=('continuous_epoch', 'min'), last_epoch=('continuous_epoch', 'max'), first_value=('value', 'first'), last_value=('value', 'last'))
        .reset_index()
        .sort_values(['support', 'support_task', 'metric'])
    )


summarize_readout(readout_tidy)


## Plot for main context

In [ ]:
# Plot for main context
def plot_metric_three_tasks(df: pd.DataFrame, metric: str, title: str, *, support: str = DEFAULT_SUPPORT, tasks: list[str] = TASKS, ax=None):
    plot_df = filter_readout(df, support=support, tasks=tasks, metrics=[metric])
    if plot_df.empty:
        available = df[['support', 'support_task', 'metric']].drop_duplicates().sort_values(['support', 'support_task', 'metric']) if not df.empty else pd.DataFrame()
        warnings.warn(f'No rows for support={support!r}, metric={metric!r}. Available combinations are displayed below.')
        display(available)
        return None

    for task in tasks:
        task_df = plot_df[plot_df['support_task'] == task].sort_values('continuous_epoch')
        if task_df.empty:
            ax.set_title(f'{task} missing')
            ax.axis('off')
            continue
        if title == "Density (1-Hoyer)":
            ax.plot(task_df['continuous_epoch'], 1-task_df['value'], marker='o', markersize=3, linewidth=1,alpha=0.7)
        else:
            ax.plot(task_df['continuous_epoch'], task_df['value'], marker='o', markersize=3, linewidth=1, label=TASK_TITLE[task],alpha=0.7)
        ax.set_xlabel('Continuous Epoch',fontsize=14)
        ax.grid(True, alpha=0.25)
    ax.set_title(title,fontsize=14)
    if title=="Trace":
        ax.legend(fontsize=14)
    fig.tight_layout()
    return fig

TITLE = ['Trace', 'eRank', 'Density (1-Hoyer)']
TASK_TITLE = {'gsm8k':'GSM8K', 'mbpp':'MBPP', 'dolly_qa':'Dolly-QA'}
fig, axes = plt.subplots(1, 3, figsize=(7, 4), sharex=True)
for i in range(3):
    metric = READOUT_METRICS[i]
    title = TITLE[i]
    plot_metric_three_tasks(readout_tidy, metric, title, support=DEFAULT_SUPPORT, ax=axes[i])
#plt.savefig('./figures/qwen0p5_SSFT.png', dpi=300, bbox_inches='tight') 
plt.show()

## Aha... This is toooooo complicated, I will switch to the wandb API to draw other figures...